# <center>Pre processing</center>
---

In [2]:
import pandas as pd
from nltk.corpus import stopwords
import nltk
import numpy as np
import os
import sys
from nltk.corpus import stopwords
nltk.download('stopwords')
import spacy as sp
from PreProcessing.pre_processing import PreProcessing


project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root_path not in sys.path:
    sys.path.append(project_root_path)

[nltk_data] Downloading package stopwords to /home/ester/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


ModuleNotFoundError: No module named 'PreProcessing'

In [3]:
df = pd.read_csv("../data/videos_info.csv")
display(df)

,video_id,title,description,channel_title,published_at,youtube_video_link,view_count,like_count,comment_count
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Joe Marsh,2024-04-10T15:05:37Z,https://www.youtube.com/watch?v=jEKzQV5oajY,1057.0,152.0,32.0
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",WWURD,2023-10-18T14:39:18Z,https://www.youtube.com/watch?v=xrGGce8cmx8,65.0,6.0,1.0
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...",The Mosaic Ark,2024-02-22T07:30:05Z,https://www.youtube.com/watch?v=uaozGpSc4nc,330.0,8.0,6.0
3,TwACgO_oPq4,Anacondaz — Акуле плевать (Official Music Video),#Anacondaz #Акулеплевать #Безпаники\n\nLong St...,ANACONDAZ,2014-05-20T09:29:45Z,https://www.youtube.com/watch?v=TwACgO_oPq4,2321870.0,24565.0,561.0
4,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,https://www.youtube.com/watch?v=A59ftbhsQUE,806.0,136.0,18.0
...,...,...,...,...,...,...,...,...,...
976570,JJMgQL2e45s,СРОЧНО💥 СМОТРЕТЬ ВСЕМ РОДИТЕЛЯМ,Визит к Васильевой в хорошем качестве на нашем...,СОВЕТ РОДИТЕЛЕЙ РОССИИ,2024-02-04T07:04:35Z,https://www.youtube.com/watch?v=JJMgQL2e45s,42381.0,4670.0,239.0
976571,CvzWVrPkEPU,Теракты в Москве. Президент Таджикистана откры...,#невзоров #новости #nevzorov\n📚Заказ книги Нев...,Александр Невзоров,2024-03-09T19:08:48Z,https://www.youtube.com/watch?v=CvzWVrPkEPU,702042.0,37337.0,2138.0
976572,uXDchAoBrrE,منصة جديدة لربح عملة الدولار usdt اول يوم عمل ...,منصة جديدة لربح عملة الدولار usdt اول يوم عمل ...,TvreK Space,2023-11-14T10:41:20Z,https://www.youtube.com/watch?v=uXDchAoBrrE,37.0,4.0,4.0
976573,gIxIevrnxhk,So gefährlich ist das #Selbstbestimmungsgesetz...,🎥 Schauen Sie sich hier den Vortrag in voller ...,DemoFürAlle,2024-11-14T11:44:59Z,https://www.youtube.com/watch?v=gIxIevrnxhk,456.0,68.0,0.0


In [ ]:
def remove_repetion_caracteres(string, max_repetition=2):
    if not string:
        return string
    
    result = string[0]
    count = 1
    
    for i in range(1, len(string)):
        if string[i] == string[i-1]:
            count += 1
            if count <= max_repetition:
                result += string[i]
        else:
            count = 1
            result += string[i]
    
    return result

def preprocess_text_pipeline(#input_csv_path='./data/dataFrame.csv', 
                              #output_csv_path='./data/dataFrame.csv',
                              df,
                              stopwords_file='stopwords.txt',
                              text_column="comments"):
   
    stem = sp.load("en_core_web_sm")
    pp = PreProcessing(language="en")
    
    custom_stopwords = [line.strip() for line in open(stopwords_file, 'r', encoding='utf-8')]
    english_stopwords = set(stopwords.words('english'))
    
    # Adiciona stopwords à lista da classe PreProcessing
    pp.append_stopwords_list(list(english_stopwords - set(pp.stopwords)) + custom_stopwords)

    def preprocessing(text):
        if pd.isna(text):
            return np.nan

        tokens = stem(text.lower()) # Processo de lematização da biblioteca spaCy - retorna a lista dos tokens do texto
        text = ' '.join([text for token in tokens for text in token.lemma_.strip().split()]) # Junta estes tokens na ordem do texto bruto
        text = pp.remove_stopwords(text) # Remove stopwords presentes
        text = pp.lowercase_unidecode(text) # Coloca tudo em lowercase e remove acento
        text = pp.remove_stopwords(text) # Remove stopwords presentes 
        text = pp.remove_tweet_marking(text) # Remove @ ou # seguido de 1 ou mais carcteres e n ' ' seguidos
        text = remove_repetion_caracteres(text) # Remove a repetição de caracteres ex: gooool -> gool
        text = pp.remove_urls(text) # Remove http\S+ *, ou seja, qualquer http seguido de 1 ou mais caracteres e os espaços no final
        text = pp.remove_punctuation(text) # Remove os sinais de pontuação e reorganiza os espaços
        text = pp.remove_numbers(text) # Remove os números
        text = pp.remove_n(text, n=3) # Remove palavras de tamanho <= n(n=3)
        
        return text

    df['clean_text'] = df[text_column].apply(preprocessing)
    return df

In [ ]:
df_clean = preprocess_text_pipeline(df=df, text_column='title')
display(df_clean)
df_clean.to_csv("../data/preprocessed_english_titles")